# 01 - Förbered och utforska

Det här notebooket bygger träningsdata med SQL-first-flöde:
- läser rådata
- gör join mot lookup-tabell
- skapar kundfeatures (RFM-liknande)
- sparar `customer_enriched.csv` för träning


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'data').exists() and (root.parent / 'data').exists():
    root = root.parent
if not (root / 'data').exists() and (root.parent.parent / 'data').exists():
    root = root.parent.parent

figures_dir = root / 'outputs' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(':memory:')

transactions_df = pd.read_csv(root / 'data' / 'raw' / 'transactions.csv', parse_dates=['InvoiceDate'])
regions_df = pd.read_csv(root / 'data' / 'raw' / 'regions.csv')

# SQLite fungerar enklast med ISO-datum som text.
transactions_df['InvoiceDate'] = transactions_df['InvoiceDate'].dt.strftime('%Y-%m-%d %H:%M:%S')
transactions_df.to_sql('transactions', conn, index=False, if_exists='replace')
regions_df.to_sql('regions', conn, index=False, if_exists='replace')

transactions_df.head()


In [ ]:
quality_df = pd.read_sql_query(
    """
    SELECT
        'transactions' AS table_name,
        COUNT(*) AS rows,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS missing_customer_id,
        SUM(CASE WHEN Country IS NULL THEN 1 ELSE 0 END) AS missing_country
    FROM transactions
    UNION ALL
    SELECT
        'regions' AS table_name,
        COUNT(*) AS rows,
        SUM(CASE WHEN Country IS NULL THEN 1 ELSE 0 END) AS missing_customer_id,
        SUM(CASE WHEN RegionGroup IS NULL THEN 1 ELSE 0 END) AS missing_country
    FROM regions
    """,
    conn,
)
quality_df


In [ ]:
customer_features_df = pd.read_sql_query(
    """
    WITH cleaned AS (
        SELECT
            t.CustomerID,
            t.Country,
            COALESCE(r.RegionGroup, 'Other') AS RegionGroup,
            t.InvoiceNo,
            t.InvoiceDate,
            t.StockCode,
            t.Quantity,
            t.UnitPrice,
            t.Quantity * t.UnitPrice AS Revenue
        FROM transactions t
        LEFT JOIN regions r
            ON t.Country = r.Country
        WHERE t.CustomerID IS NOT NULL
          AND t.Quantity > 0
          AND t.UnitPrice > 0
    ),
    invoice_level AS (
        SELECT
            CustomerID,
            InvoiceNo,
            SUM(Revenue) AS invoice_total
        FROM cleaned
        GROUP BY CustomerID, InvoiceNo
    ),
    customer_base AS (
        SELECT
            CustomerID,
            MAX(InvoiceDate) AS last_purchase_date,
            COUNT(DISTINCT InvoiceNo) AS frequency,
            SUM(Revenue) AS monetary,
            SUM(Quantity) AS basket_size,
            COUNT(DISTINCT StockCode) AS unique_products,
            AVG(UnitPrice) AS avg_unit_price,
            MAX(Country) AS Country,
            MAX(RegionGroup) AS RegionGroup
        FROM cleaned
        GROUP BY CustomerID
    ),
    ref_date AS (
        SELECT MAX(InvoiceDate) AS ref_date
        FROM cleaned
    )
    SELECT
        cb.CustomerID,
        cb.Country,
        cb.RegionGroup,
        cb.last_purchase_date,
        CAST(julianday((SELECT ref_date FROM ref_date)) - julianday(cb.last_purchase_date) AS INTEGER) AS recency_days,
        cb.frequency,
        cb.monetary,
        inv.avg_order_value,
        cb.basket_size,
        CAST(cb.basket_size * 1.0 / cb.frequency AS REAL) AS avg_items_per_invoice,
        cb.unique_products,
        cb.avg_unit_price
    FROM customer_base cb
    JOIN (
        SELECT CustomerID, AVG(invoice_total) AS avg_order_value
        FROM invoice_level
        GROUP BY CustomerID
    ) inv
    USING (CustomerID)
    ORDER BY cb.CustomerID
    """,
    conn,
)

customer_features_df.to_csv(root / 'data' / 'raw' / 'customers.csv', index=False)
customer_features_df.to_csv(root / 'data' / 'processed' / 'customer_enriched.csv', index=False)
customer_features_df.head()


In [ ]:
missing_values_df = customer_features_df.isna().sum().sort_values(ascending=False).rename('missing_values').to_frame()
missing_values_df


In [ ]:
numeric_features = [
    'recency_days',
    'frequency',
    'monetary',
    'avg_order_value',
    'basket_size',
    'avg_items_per_invoice',
    'unique_products',
    'avg_unit_price',
]

plt.figure(figsize=(10, 7))
sns.heatmap(customer_features_df[numeric_features].corr(numeric_only=True), annot=True, fmt='.2f', cmap='Blues', center=0)
plt.title('Korrelation mellan numeriska features')
plt.tight_layout()
plt.savefig(figures_dir / 'correlation_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()


Output från notebook 1:
- `data/processed/customer_enriched.csv`
- `outputs/figures/correlation_heatmap.png`
